# CALSNIC exact-SDF cortical reconstruction: PCA vs multiresolution INR

This notebook renders **indexed triangle meshes**, not point clouds. It compares the physical-mm target with the matched train-only PCA oracle and the completed multiresolution evaluations. Validation contains MR64 and MR128; locked test evaluation was produced only for the selected MR128 model and is shown at marching-cubes resolutions 256 and 512.

The notebook also reports paired metrics, topology, overlays, and two directions of exact point-to-triangle error. PCA rank 172 is an oracle projection: its coefficients are computed from the complete target mesh, whereas INR uses a code fitted to held-out SDF samples.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import point_cloud_utils as pcu
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import trimesh

ROOT = Path('/mnt/bulk10tb/Deep3DComp/CALSNIC/control_L_exact_multires_v1')
MANIFEST_PATH = ROOT / 'manifests/calsnic_control_L_exact.csv'

VAL_EVALUATIONS = {
    'MR64 model / MC256': ROOT / 'runs/calsnic_control_L_multires64_z256_exact/manual_evaluation/best_mesh',
    'MR128 model / MC256': ROOT / 'runs/calsnic_control_L_multires128_z256_exact/manual_evaluation/best_mesh',
}
TEST_EVALUATIONS = {
    'MR128 model / MC256': ROOT / 'final_test/multires128_best_mesh_r256',
    'MR128 model / MC512': ROOT / 'final_test/multires128_best_mesh_r512',
}
EVALUATIONS = {'val': VAL_EVALUATIONS, 'test': TEST_EVALUATIONS}

# Set either ID to a particular scan string. None selects the median-ASSD case.
VAL_SCAN_ID = None
TEST_SCAN_ID = None
EXAMPLE_MODE = 'median'  # one of: best, median, worst
ERROR_COLOR_MAX_MM = 5.0
PLOT_FACE_LIMIT = 350_000  # None uses every face; a limit affects rendering only.

assert ROOT.is_dir(), ROOT
assert MANIFEST_PATH.is_file(), MANIFEST_PATH
manifest = pd.read_csv(MANIFEST_PATH).set_index('scan_id')

## Aggregate validation and test results

Lower is better for ASSD, HD95, high-curvature distance, and component count; higher is better for F-score and normal cosine. Connected-component count is a topology diagnostic and is not part of ASSD.

In [ ]:
metric_frames = []
for split, evaluations in EVALUATIONS.items():
    for evaluation, directory in evaluations.items():
        path = directory / 'per_scan_metrics.csv'
        if not path.is_file():
            print(f'Missing evaluation: {path}')
            continue
        frame = pd.read_csv(path)
        frame = frame.query('split == @split').copy()
        frame['evaluation'] = evaluation
        metric_frames.append(frame)

metrics = pd.concat(metric_frames, ignore_index=True)
SUMMARY_METRICS = [
    'assd_mm', 'hd95_mm', 'gt_to_prediction_mm', 'prediction_to_gt_mm',
    'high_curvature_gt_to_prediction_mm', 'normal_absolute_cosine',
    'fscore_1mm', 'volume_relative_error', 'predicted_connected_components',
]
summary = (
    metrics.groupby(['split', 'evaluation', 'method'])[SUMMARY_METRICS]
    .agg(['mean', 'median'])
    .round(4)
)
display(summary)

In [ ]:
plot_frame = metrics.query('method == "inr"').copy()
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['ASSD (mm, lower is better)', 'HD95 (mm, lower is better)', 'Connected components (1 expected)'],
)
for split, color in [('val', '#4477AA'), ('test', '#EE7733')]:
    part = plot_frame.query('split == @split')
    for column, metric in enumerate(['assd_mm', 'hd95_mm', 'predicted_connected_components'], start=1):
        fig.add_trace(
            go.Box(x=part['evaluation'], y=part[metric], name=split, legendgroup=split,
                   marker_color=color, boxpoints='all', jitter=0.25, pointpos=0,
                   showlegend=(column == 1)),
            row=1, col=column,
        )
fig.update_xaxes(tickangle=20)
fig.update_layout(height=520, width=1500, title='INR geometry and topology distributions')
fig.show()

## Mesh-loading and rendering helpers

If `PLOT_FACE_LIMIT` is set, very dense meshes use a deterministic face subset only for browser rendering. All metrics and error values still use the complete meshes. Increase the limit or set it to `None` for full MC512 rendering.

In [ ]:
def load_mesh(path):
    mesh = trimesh.load(Path(path), process=False)
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    if not isinstance(mesh, trimesh.Trimesh) or not len(mesh.faces):
        raise ValueError(f'Invalid mesh: {path}')
    return mesh

def component_labels(mesh):
    # Label faces directly; this avoids constructing hundreds of submeshes.
    return trimesh.graph.connected_component_labels(
        mesh.face_adjacency, node_count=len(mesh.faces)
    )

def component_count(mesh):
    labels = component_labels(mesh)
    return int(labels.max() + 1) if len(labels) else 0

def plot_arrays(mesh, face_limit=PLOT_FACE_LIMIT):
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    if face_limit is not None and len(faces) > face_limit:
        step = math.ceil(len(faces) / face_limit)
        faces = faces[::step]
    return vertices, faces

def mesh_trace(mesh, name, color=None, intensity=None, opacity=1.0, showscale=False):
    vertices, faces = plot_arrays(mesh)
    kwargs = dict(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        name=name, opacity=opacity, flatshading=False, showscale=showscale,
        lighting=dict(ambient=0.45, diffuse=0.8, specular=0.15, roughness=0.8),
    )
    if intensity is None:
        kwargs['color'] = color
    else:
        kwargs.update(
            intensity=np.asarray(intensity), intensitymode='vertex', colorscale='Turbo',
            cmin=0.0, cmax=ERROR_COLOR_MAX_MM,
            colorbar=dict(title='mm', len=0.7),
        )
    return go.Mesh3d(**kwargs)

def exact_vertex_to_mesh_distance(source, target):
    distance, _, _ = pcu.closest_points_on_mesh(
        np.asarray(source.vertices, dtype=np.float64),
        np.asarray(target.vertices, dtype=np.float64),
        np.asarray(target.faces, dtype=np.int32),
    )
    return np.asarray(distance)

def available_scan_ids(split):
    evaluations = EVALUATIONS[split]
    sets = []
    for directory in evaluations.values():
        sets.append({path.stem for path in (directory / 'meshes/inr' / split).glob('*.ply')})
    return sorted(set.intersection(*sets))

def choose_scan(split, explicit=None, mode=EXAMPLE_MODE):
    available = available_scan_ids(split)
    if explicit is not None:
        if explicit not in available:
            raise KeyError(f'{explicit} is not available for every {split} evaluation')
        return explicit
    primary = list(EVALUATIONS[split])[-1]
    frame = metrics.query(
        'split == @split and method == "inr" and evaluation == @primary and scan_id in @available'
    ).sort_values('assd_mm')
    position = {'best': 0, 'median': len(frame) // 2, 'worst': len(frame) - 1}[mode]
    return str(frame.iloc[position]['scan_id'])

def load_case(split, scan_id):
    target = load_mesh(manifest.loc[scan_id, 'mesh_path_mm'])
    reconstructions = {}
    pca = None
    for label, directory in EVALUATIONS[split].items():
        inr_path = directory / 'meshes/inr' / split / f'{scan_id}.ply'
        if inr_path.is_file():
            reconstructions[label] = load_mesh(inr_path)
        pca_path = directory / 'meshes/pca' / split / f'{scan_id}.ply'
        if pca is None and pca_path.is_file():
            pca = load_mesh(pca_path)
    if pca is None:
        raise FileNotFoundError(f'No PCA mesh for {split}/{scan_id}')
    return target, pca, reconstructions

val_scan = choose_scan('val', VAL_SCAN_ID)
test_scan = choose_scan('test', TEST_SCAN_ID)
print('Validation example:', val_scan)
print('Test example:', test_scan)

In [ ]:
def selected_metric_table(split, scan_id):
    columns = ['evaluation', 'method', 'assd_mm', 'hd95_mm', 'gt_to_prediction_mm',
               'prediction_to_gt_mm', 'high_curvature_gt_to_prediction_mm',
               'normal_absolute_cosine', 'fscore_1mm', 'volume_relative_error',
               'predicted_connected_components']
    return metrics.query('split == @split and scan_id == @scan_id')[columns].round(4)

print(f'Validation: {val_scan}')
display(selected_metric_table('val', val_scan))
print(f'Test: {test_scan}')
display(selected_metric_table('test', test_scan))

## Target vs PCA vs multiresolution triangle meshes

These panels retain triangle connectivity. The camera can be rotated and zoomed independently. Matching the overall silhouette is not sufficient: inspect deep sulci, gyral crowns, detached sheets, and small closed components.

In [ ]:
def show_mesh_panel(split, scan_id):
    target, pca, inr = load_case(split, scan_id)
    meshes = {'Target': target, 'PCA oracle rank 172': pca, **inr}
    colors = ['#999999', '#4477AA', '#EE7733', '#228833', '#CC6677']
    subtitles = []
    for name, mesh in meshes.items():
        subtitles.append(f'{name}<br>V={len(mesh.vertices):,}, F={len(mesh.faces):,}, CC={component_count(mesh)}')
    fig = make_subplots(
        rows=1, cols=len(meshes), specs=[[{'type': 'scene'}] * len(meshes)],
        subplot_titles=subtitles, horizontal_spacing=0.01,
    )
    for column, ((name, mesh), color) in enumerate(zip(meshes.items(), colors), start=1):
        fig.add_trace(mesh_trace(mesh, name, color=color), row=1, col=column)
        scene = 'scene' if column == 1 else f'scene{column}'
        fig.update_layout(**{scene: dict(
            aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
            camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)),
        )})
    fig.update_layout(height=670, width=440 * len(meshes), title=f'{split.upper()} triangle meshes — {scan_id}')
    fig.show()
    return meshes

val_meshes = show_mesh_panel('val', val_scan)

In [ ]:
test_meshes = show_mesh_panel('test', test_scan)

## PCA vs SPHARM (different degrees) triangle meshes

Reconstructions from `task_exact_spharm_cortex_v1`, evaluated against the same PCA rank-172 oracle
basis used above. Two SPHARM degrees are shown side by side with PCA and the target: degree 16
(867 coefficients — matched to PCA's raw corresponded-vertex RMSE ceiling) and degree 32 (3267
coefficients — well past PCA's rank-172 cap, since PCA's rank cannot exceed `train_scans - 1`
while SPHARM's degree is not limited by cohort size). Only the validation split has been evaluated
here (the test split is locked behind `--confirm-test` and is not run just to populate a notebook).

Briefly, from that task's `README.md`: at matched RMSE (degree 16) SPHARM loses on every surface
metric shown below (Gibbs-like ringing concentrated near high curvature); only at degree 32 does it
decisively beat PCA on every metric, on 14/15 validation scans.

In [ ]:
SPHARM_ROOT = ROOT / 'comparisons'
SPHARM_EVALUATIONS = {
    'val': {
        'SPHARM degree 16 (867 coef)': SPHARM_ROOT / 'spharm_degree16_vs_pca172',
        'SPHARM degree 32 (3267 coef)': SPHARM_ROOT / 'spharm_degree32_vs_pca172',
    },
}

spharm_metric_frames = []
for split, evaluations in SPHARM_EVALUATIONS.items():
    for evaluation, directory in evaluations.items():
        path = directory / 'per_scan_metrics.csv'
        if not path.is_file():
            print(f'Missing SPHARM evaluation: {path}')
            continue
        frame = pd.read_csv(path)
        frame = frame.query('split == @split').copy()
        frame['evaluation'] = evaluation
        spharm_metric_frames.append(frame)

spharm_metrics = pd.concat(spharm_metric_frames, ignore_index=True)
spharm_summary = (
    spharm_metrics.groupby(['split', 'evaluation', 'method'])[SUMMARY_METRICS]
    .agg(['mean', 'median'])
    .round(4)
)
display(spharm_summary)

In [ ]:
def load_spharm_case(split, scan_id):
    target = load_mesh(manifest.loc[scan_id, 'mesh_path_mm'])
    pca = None
    spharm = {}
    for label, directory in SPHARM_EVALUATIONS[split].items():
        spharm_path = directory / 'meshes/spharm' / split / f'{scan_id}.ply'
        if spharm_path.is_file():
            spharm[label] = load_mesh(spharm_path)
        pca_path = directory / 'meshes/pca' / split / f'{scan_id}.ply'
        if pca is None and pca_path.is_file():
            pca = load_mesh(pca_path)
    if pca is None:
        raise FileNotFoundError(f'No PCA mesh for {split}/{scan_id}')
    return target, pca, spharm

def show_spharm_panel(split, scan_id):
    target, pca, spharm = load_spharm_case(split, scan_id)
    meshes = {'Target': target, 'PCA oracle rank 172': pca, **spharm}
    colors = ['#999999', '#4477AA', '#EE7733', '#CC6677']
    subtitles = []
    for name, mesh in meshes.items():
        subtitles.append(f'{name}<br>V={len(mesh.vertices):,}, F={len(mesh.faces):,}, CC={component_count(mesh)}')
    fig = make_subplots(
        rows=1, cols=len(meshes), specs=[[{'type': 'scene'}] * len(meshes)],
        subplot_titles=subtitles, horizontal_spacing=0.01,
    )
    for column, ((name, mesh), color) in enumerate(zip(meshes.items(), colors), start=1):
        fig.add_trace(mesh_trace(mesh, name, color=color), row=1, col=column)
        scene = 'scene' if column == 1 else f'scene{column}'
        fig.update_layout(**{scene: dict(
            aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
            camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)),
        )})
    fig.update_layout(height=670, width=440 * len(meshes), title=f'{split.upper()} PCA vs SPHARM — {scan_id}')
    fig.show()
    return meshes

spharm_val_meshes = show_spharm_panel('val', val_scan)

In [ ]:
def selected_spharm_metric_table(split, scan_id):
    columns = ['evaluation', 'method', 'assd_mm', 'hd95_mm', 'gt_to_prediction_mm',
               'prediction_to_gt_mm', 'high_curvature_gt_to_prediction_mm',
               'normal_absolute_cosine', 'fscore_1mm', 'volume_relative_error',
               'predicted_connected_components']
    return spharm_metrics.query('split == @split and scan_id == @scan_id')[columns].round(4)

print(f'Validation: {val_scan}')
display(selected_spharm_metric_table('val', val_scan))

## Two-sided exact surface-error maps

The first direction colors reconstruction vertices by distance to the target (precision-like). The second colors target vertices by distance to the reconstruction (coverage/recall-like). The common color range is clipped at `ERROR_COLOR_MAX_MM`; clipping affects color only. The asymmetry is important here because the INR often places predicted surface near the cortex while failing to cover parts of the true cortical surface continuously.

In [ ]:
def show_error_maps(split, scan_id):
    target, pca, inr = load_case(split, scan_id)
    reconstructions = {'PCA oracle rank 172': pca, **inr}
    fig = make_subplots(
        rows=2, cols=len(reconstructions),
        specs=[[{'type': 'scene'}] * len(reconstructions)] * 2,
        subplot_titles=(
            [f'{name}: reconstruction → target' for name in reconstructions]
            + [f'{name}: target → reconstruction' for name in reconstructions]
        ),
        vertical_spacing=0.05, horizontal_spacing=0.01,
    )
    for column, (name, reconstruction) in enumerate(reconstructions.items(), start=1):
        reconstruction_error = exact_vertex_to_mesh_distance(reconstruction, target)
        target_error = exact_vertex_to_mesh_distance(target, reconstruction)
        fig.add_trace(mesh_trace(reconstruction, name, intensity=reconstruction_error,
                                 showscale=(column == len(reconstructions))), row=1, col=column)
        fig.add_trace(mesh_trace(target, name, intensity=target_error,
                                 showscale=False), row=2, col=column)
        for row in (1, 2):
            index = (row - 1) * len(reconstructions) + column
            scene = 'scene' if index == 1 else f'scene{index}'
            fig.update_layout(**{scene: dict(
                aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
                camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)),
            )})
    fig.update_layout(height=1120, width=500 * len(reconstructions),
                      title=f'{split.upper()} exact point-to-triangle error — {scan_id}')
    fig.show()

show_error_maps('val', val_scan)

In [ ]:
show_error_maps('test', test_scan)

## Target/reconstruction overlays

Blue is the target and orange is the reconstruction. Toggle legend entries to isolate a method. Semi-transparency makes systematic offsets and detached surfaces easier to see than separate panels.

In [ ]:
def show_overlays(split, scan_id):
    target, pca, inr = load_case(split, scan_id)
    reconstructions = {'PCA oracle rank 172': pca, **inr}
    fig = make_subplots(
        rows=1, cols=len(reconstructions), specs=[[{'type': 'scene'}] * len(reconstructions)],
        subplot_titles=list(reconstructions), horizontal_spacing=0.01,
    )
    for column, (name, reconstruction) in enumerate(reconstructions.items(), start=1):
        fig.add_trace(mesh_trace(target, 'Target', color='#4477AA', opacity=0.45), row=1, col=column)
        fig.add_trace(mesh_trace(reconstruction, name, color='#EE7733', opacity=0.50), row=1, col=column)
        scene = 'scene' if column == 1 else f'scene{column}'
        fig.update_layout(**{scene: dict(
            aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
            camera=dict(eye=dict(x=1.4, y=1.4, z=0.9)),
        )})
    fig.update_layout(height=700, width=520 * len(reconstructions),
                      title=f'{split.upper()} overlays — {scan_id}', showlegend=True)
    fig.show()

show_overlays('val', val_scan)
show_overlays('test', test_scan)

## Fragmentation diagnostic

The target and PCA meshes have one connected component. The raw INR fields produce many components. The largest component contains most predicted surface area, but that does **not** imply that component filtering fixes the reconstruction: in a 10,000-point test diagnostic, retaining only the largest MR128/MC256 component worsened mean ASSD from about 1.664 mm to 1.943 mm and HD95 from 7.196 mm to 9.663 mm. Detached pieces are filling regions missed by the main surface, so fragmentation is a learned-field problem rather than a cosmetic export artifact.

In [ ]:
def fragmentation_table(split, scan_id):
    target, pca, inr = load_case(split, scan_id)
    rows = []
    for name, mesh in {'Target': target, 'PCA oracle rank 172': pca, **inr}.items():
        labels = component_labels(mesh)
        areas = np.bincount(labels, weights=np.asarray(mesh.area_faces))
        rows.append({
            'method': name,
            'connected_components': len(areas),
            'largest_component_area_fraction': areas.max() / areas.sum(),
            'components_area_gt_1_mm2': int(np.sum(areas > 1.0)),
            'components_area_gt_10_mm2': int(np.sum(areas > 10.0)),
            'total_area_mm2': mesh.area,
        })
    return pd.DataFrame(rows).set_index('method')

display(fragmentation_table('val', val_scan).round(4))
display(fragmentation_table('test', test_scan).round(4))

## Training behavior and capacity

MR128 has about 8× as many shared grid parameters as MR64 (13.62 M vs 1.70 M), but both use the same 256-D per-scan code and 128-wide MLP. This cell shows sampled training losses and held-out SDF loss. Mesh quality must still be assessed separately: the checkpoint with minimum held-out SDF L1 occurred much earlier than the best mesh checkpoint.

In [ ]:
RUNS = {
    'MR64': ROOT / 'runs/calsnic_control_L_multires64_z256_exact',
    'MR128': ROOT / 'runs/calsnic_control_L_multires128_z256_exact',
}
fig = make_subplots(rows=1, cols=2, subplot_titles=['Training SDF L1 (mean of broad and near)', 'Held-out fitted-code SDF L1'])
capacity_rows = []
for label, run in RUNS.items():
    history = pd.read_csv(run / 'logs/training_history.csv')
    validation = pd.read_csv(run / 'logs/validation_history.csv')
    color = '#4477AA' if label == 'MR64' else '#EE7733'
    fig.add_trace(go.Scatter(x=history.epoch, y=0.5 * (history.broad_sdf_l1 + history.near_sdf_l1),
                             name=f'{label} train', line=dict(color=color)), row=1, col=1)
    fig.add_trace(go.Scatter(x=validation.epoch, y=validation.value, name=f'{label} held-out',
                             line=dict(color=color)), row=1, col=2)
    resolutions = (
        [8, 16, 24, 32, 48, 64] if label == 'MR64'
        else [8, 16, 24, 32, 48, 64, 96, 128]
    )
    grid_parameters = 4 * sum(resolution ** 3 for resolution in resolutions)
    capacity_rows.append({'model': label, 'grid_levels': str(resolutions),
                          'shared_grid_parameters': grid_parameters,
                          'best_heldout_sdf_l1': validation.value.min(),
                          'final_heldout_sdf_l1': validation.iloc[-1].value})
fig.update_layout(height=500, width=1250, title='Training/held-out loss is not the same objective as mesh fidelity')
fig.update_yaxes(title_text='normalized SDF L1', row=1, col=1)
fig.update_yaxes(title_text='normalized SDF L1', row=1, col=2)
fig.show()
display(pd.DataFrame(capacity_rows).set_index('model'))

## Interpretation recorded with this run

- **MR128 is the better learned model on validation geometry:** versus MR64, mean ASSD fell 11.4% (1.888 → 1.673 mm), HD95 fell 17.2%, and high-curvature target-to-prediction error fell 15.0%; every validation scan improved on these three distances. MR128 nevertheless had worse normals and much more fragmentation.
- **MC512 is a finer measurement of the same MR128 field, not a higher-capacity model.** On test it reduced ASSD 5.6% and HD95 8.8%, but worsened normal cosine and doubled component count. It reveals and localizes existing zero crossings; it cannot repair the learned field.
- **PCA remains better for overall surface geometry.** On test, PCA ASSD was 1.335 mm versus 1.570 mm for MR128/MC512, and PCA HD95 was 4.014 mm versus 6.527 mm. INR was better for relative volume error and prediction-to-target distance, but its target-to-prediction error, normals, and topology were worse.
- **PCA is an optimistic oracle baseline.** Its rank-172 coefficients are projected from each complete target mesh. The INR code is fitted from held-out SDF samples. The comparison is useful for attainable reconstruction fidelity but is not an equal-input inference comparison.
- **The immediate limitation is field regularity and subject-code saturation, not simply grid size.** Both trainings used zero Eikonal and zero TV weight; MR128 test/validation code norms hit the hard 1.0 bound, as did almost all MR128 training codes. More dense-grid capacity alone is therefore not the next controlled experiment.